# GemmaFit GGUF Artifact Validation

Validate the GGUF files produced by `finetune/train_gemma4_pipeline.ipynb`.

Default target: `/content/drive/MyDrive/GemmaFit_train/gemmafit-v2-q4_k_m.gguf`.

This notebook checks:

- Drive file exists and is non-empty
- GGUF magic/header is valid
- llama.cpp can build `llama-cli`
- llama.cpp can load the GGUF and run a short prompt

Override the model path before running with:

```python
import os
os.environ["GEMMAFIT_GGUF"] = "/content/drive/MyDrive/GemmaFit_train/gemmafit-v2-q5_k_m.gguf"
```


In [ ]:
# Runtime and Drive setup.
import json
import os
import sys
import time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

WORKDIR = Path(os.environ.get('GEMMAFIT_WORKDIR', '/content/drive/MyDrive/GemmaFit_train'))
MODEL_PATH = Path(os.environ.get('GEMMAFIT_GGUF', str(WORKDIR / 'gemmafit-v2-q4_k_m.gguf')))
OPTIONAL_MODEL_PATHS = [
    WORKDIR / 'gemmafit-v2-q5_k_m.gguf',
]
REPORT_PATH = Path(os.environ.get(
    'GEMMAFIT_GGUF_REPORT',
    str(WORKDIR / 'gemmafit_gguf_validation_report.json'),
))

print('Python:', sys.version)
print('Workdir:', WORKDIR)
print('Target GGUF:', MODEL_PATH)
print('Report:', REPORT_PATH)


In [ ]:
# File-level integrity checks.
import hashlib

MIN_GGUF_BYTES = int(os.environ.get('MIN_GGUF_BYTES', str(1024 ** 3)))
COMPUTE_SHA256 = os.environ.get('COMPUTE_SHA256', '0') == '1'

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'GGUF file not found: {MODEL_PATH}')
if not MODEL_PATH.is_file():
    raise IsADirectoryError(f'GGUF path is not a file: {MODEL_PATH}')

size_bytes = MODEL_PATH.stat().st_size
if size_bytes < MIN_GGUF_BYTES:
    raise RuntimeError(
        f'GGUF looks too small: {size_bytes} bytes < {MIN_GGUF_BYTES} bytes. '
        'Check whether conversion or quantization failed.'
    )

with MODEL_PATH.open('rb') as f:
    header = f.read(8)
if header[:4] != b'GGUF':
    raise ValueError(f'Invalid GGUF magic bytes: {header[:4]!r}')
gguf_version = int.from_bytes(header[4:8], byteorder='little', signed=False)
if gguf_version < 2:
    raise ValueError(f'Unexpected GGUF version: {gguf_version}')

sha256 = None
if COMPUTE_SHA256:
    h = hashlib.sha256()
    with MODEL_PATH.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    sha256 = h.hexdigest()

optional = []
for path in OPTIONAL_MODEL_PATHS:
    optional.append({
        'path': str(path),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else None,
    })

report = {
    'checked_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model_path': str(MODEL_PATH),
    'size_bytes': size_bytes,
    'size_gib': round(size_bytes / 1024 ** 3, 3),
    'gguf_version': gguf_version,
    'sha256': sha256,
    'optional_models': optional,
}

print(json.dumps(report, indent=2))


In [ ]:
# Fetch and build llama.cpp for runtime validation.
import shutil
import subprocess

LLAMA_CPP_DIR = Path(os.environ.get('LLAMA_CPP_DIR', '/content/llama.cpp'))
LLAMA_CPP_URL = os.environ.get('LLAMA_CPP_URL', 'https://github.com/ggerganov/llama.cpp')
shutil.rmtree(LLAMA_CPP_DIR, ignore_errors=True)

subprocess.run([
    'git', 'clone', '--depth', '1', LLAMA_CPP_URL, str(LLAMA_CPP_DIR),
], check=True)
subprocess.run(['cmake', '-B', 'build', '-DGGML_CUDA=OFF'], cwd=LLAMA_CPP_DIR, check=True)
subprocess.run([
    'cmake', '--build', 'build', '--target', 'llama-cli', '-j',
], cwd=LLAMA_CPP_DIR, check=True)

LLAMA_CLI = LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-cli'
if not LLAMA_CLI.exists():
    raise FileNotFoundError(f'llama-cli was not built: {LLAMA_CLI}')

version = subprocess.run(
    [str(LLAMA_CLI), '--version'],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
).stdout.strip()
report['llama_cpp_version'] = version
print(version)


In [ ]:
# Load-test the GGUF with a short deterministic prompt.
PROMPT = os.environ.get('VALIDATION_PROMPT', 'Return the word READY only.')
N_PREDICT = int(os.environ.get('VALIDATION_N_PREDICT', '16'))
CTX_SIZE = int(os.environ.get('VALIDATION_CTX_SIZE', '2048'))
TIMEOUT_SEC = int(os.environ.get('VALIDATION_TIMEOUT_SEC', '900'))

cmd = [
    str(LLAMA_CLI),
    '-m', str(MODEL_PATH),
    '-p', PROMPT,
    '-n', str(N_PREDICT),
    '-c', str(CTX_SIZE),
    '--temp', '0',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=TIMEOUT_SEC,
)
print(result.stdout[-6000:])
if result.returncode != 0:
    raise RuntimeError(f'llama-cli failed with exit code {result.returncode}')

report['load_test'] = {
    'returncode': result.returncode,
    'prompt': PROMPT,
    'n_predict': N_PREDICT,
    'ctx_size': CTX_SIZE,
    'output_tail': result.stdout[-2000:],
}
print('GGUF load test passed.')


In [ ]:
# Persist validation report to Drive.
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Validation report written to:', REPORT_PATH)
print(json.dumps(report, indent=2)[:4000])
